# GDS Import, Generation, Export, And EMX Port Mapping

This public notebook shows the portable GDS side of an `emx-dbs` study. It covers three common tasks:

- Import and inspect an existing GDS file.
- Generate a custom parametric GDS seed, using the dual-core VCO tank as the example.
- Export a configured square-pixel candidate GDS and understand the EMX port/proc/command mapping.

All generated files are written under `local/notebooks/gds_import_generation_export_emx_ports/`, which is ignored by git. The notebook uses the fake EMX backend by default so it can run on any machine without Cadence/EMX installed.


In [ ]:
from __future__ import annotations

import json
import shlex
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Image, display

# Find the source checkout even when Jupyter starts in notebooks/.
repo_root = Path.cwd().resolve()
for candidate in (repo_root, *repo_root.parents):
    if (candidate / 'pyproject.toml').exists() and (candidate / 'emx_dbs').exists():
        repo_root = candidate
        break
else:
    raise RuntimeError('Could not find the emx-dbs repository root')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

out_dir = repo_root / 'local' / 'notebooks' / 'gds_import_generation_export_emx_ports'
out_dir.mkdir(parents=True, exist_ok=True)

print('Repository root:', repo_root)
print('Notebook output:', out_dir)


In [ ]:
from emx_dbs.config import load_config, prepare_run_dir
from emx_dbs.dbs import eval_one
from emx_dbs.emx_runner import frequency_sweep_ghz, validate_emx_environment
from emx_dbs.gds_io import inspect_gds, inspect_raw_gds, write_candidate_gds
from emx_dbs.rasterize import rasterize_config
from emx_dbs.reporting import generate_report, write_gds_preview, write_layout_preview
from emx_dbs.tank_generator import (
    DualCoreVcoTankGeometry,
    generate_dual_core_vco_tank_gds,
    write_dual_core_vco_tank_config,
)


## 1. Import And Inspect An Existing GDS

Start by pointing `INPUT_GDS` and `INPUT_TOP_CELL` at any layout you want to inspect. The default uses the small public example seed in this repository. `inspect_raw_gds` does not need an optimizer config; it reports cells, layers, polygon counts, label counts, bounding boxes, and area by raw GDS layer/datatype.


In [ ]:
INPUT_GDS = repo_root / 'examples' / 'generic_nport' / 'seed.gds'
INPUT_TOP_CELL = 'TOP'

raw_input_info = inspect_raw_gds(INPUT_GDS, top_cell=INPUT_TOP_CELL)
print(json.dumps(raw_input_info, indent=2, sort_keys=True))

input_preview = out_dir / 'input_gds_preview.png'
write_gds_preview(INPUT_GDS, input_preview, top_cell=INPUT_TOP_CELL, show_legend=True, show_title=True)
display(Image(filename=str(input_preview)))


## 2. Generate A Custom Dual-Core VCO Tank GDS

The generator below creates a square-pixel dual-core VCO tank seed. The physical layer defaults are N16-oriented examples: M9 on GDS `39/60`, M8 on `38/40`, V8 on `58/60`, and an optional lower-metal guard ring on `31/0`. For another process, change these GDS layer/datatype pairs to match the EMX process file used by that environment.

The port labels follow a differential tank convention:

- `PP`, `PN`: primary positive/negative feed labels.
- `SP`, `SN`: secondary positive/negative feed labels.
- `GS`, `GN`: optional guard or ground reference labels when guard ports are enabled.

The names are ordinary GDS text labels. Keep the names in the GDS, YAML `ports` list, objective parameters, and EMX command aligned.


In [ ]:
geom = DualCoreVcoTankGeometry(
    top_cell='public_dual_core_vco_tank_ring33',
    core_width_um=330.0,
    core_height_um=330.0,
    pitch_um=10.0,
    m9_ring_width_um=20.0,
    center_gap_um=10.0,
    feed_spacing_um=20.0,
    feed_width_um=5.0,
    feed_length_um=26.0,
    m8_trace_width_um=10.0,
    pixel_style='square',
    emit_v8=True,
    include_guard_ring=True,
    guard_layer=(31, 0),
    guard_offset_um=26.0,
    guard_width_um=10.0,
    guard_feed_overlap_um=5.0,
    include_guard_ports=True,
)

seed_gds = out_dir / 'public_dual_core_vco_tank_ring33.gds'
config_path = out_dir / 'public_dual_core_vco_tank_ring33.local.yaml'
raw_seed_gds = generate_dual_core_vco_tank_gds(seed_gds, geom)
write_dual_core_vco_tank_config(
    config_path,
    raw_seed_gds,
    geom,
    run_id='public_dual_core_vco_tank_ring33_demo',
    output_root=out_dir / 'runs',
)

print('Generated seed GDS:', seed_gds)
print('Generated local config:', config_path)
print(json.dumps(inspect_raw_gds(seed_gds, top_cell=geom.top_cell), indent=2, sort_keys=True))

seed_preview = out_dir / 'generated_tank_raw_preview.png'
write_gds_preview(seed_gds, seed_preview, top_cell=geom.top_cell, show_legend=True, show_title=True)
display(Image(filename=str(seed_preview)))


## 3. Read The Generated `emx-dbs` Config

The YAML config is the contract between GDS, pixelization, EMX, and the objective. Logical layer names such as `m9`, `m8`, and `v8` are local handles used by `emx-dbs`; EMX itself only sees the GDS layer/datatype numbers and the process file. The process file must define those GDS layers as the intended stack metals/vias.


In [ ]:
cfg = load_config(config_path)
print(json.dumps(validate_emx_environment(cfg), indent=2, sort_keys=True, default=str))
print(json.dumps(inspect_gds(cfg), indent=2, sort_keys=True, default=str))

layer_table = pd.DataFrame(
    [
        {'logical_layer': name, 'gds_layer': spec[0], 'gds_datatype': spec[1]}
        for name, spec in cfg.layers.items()
    ]
)
port_table = pd.DataFrame(
    [
        {
            'touchstone_index': idx,
            'name': port.name,
            'logical_layer': port.layer,
            'xy_um': tuple(port.xy_um),
            'edge': port.edge,
            'width_um': port.width_um,
        }
        for idx, port in enumerate(cfg.ports, start=1)
    ]
)

print('Layer mapping used by emx-dbs and expected by the EMX proc file:')
display(layer_table)
print('Port order used for result.sNp and objective port numbers:')
display(port_table)


## 4. Rasterize The Seed And Export A Candidate GDS

Rasterization turns the configured GDS layers inside mutable windows into boolean masks. Exporting writes a new GDS from those masks. This is the same path used before each EMX evaluation.

Corner-overlap bridge support is controlled by the config DRC fields `allow_same_layer_diagonal_contact` and `corner_overlap_bridge`. When enabled, diagonal-only same-layer pixel contacts are treated as connected and exported with small bridge polygons.


In [ ]:
maskset = rasterize_config(cfg)
run_dir = prepare_run_dir(cfg, config_path)
preview_path = out_dir / 'rasterized_layout_preview.png'
exported_gds = out_dir / 'exported_square_candidate.gds'

write_layout_preview(maskset, preview_path, cfg, annotate_geometry=True, show_legend=True, show_title=True)
write_candidate_gds(maskset, cfg, exported_gds)

print('Rasterized preview:', preview_path)
print('Exported candidate GDS:', exported_gds)
print(json.dumps(inspect_raw_gds(exported_gds, top_cell=cfg.layout.top_cell), indent=2, sort_keys=True))
display(Image(filename=str(preview_path)))


## 5. EMX Port And Process-File Mapping

`emx-dbs` builds EMX internal-port flags from the YAML `ports` list. For each port with `width_um`, the generated EMX script contains an option like `--internal=PP,5`. The labels must exist in the exported GDS on the configured port layer.

The EMX process file is passed as the final process argument after the GDS path and top-cell name. It is responsible for mapping raw GDS layer/datatype pairs to the real metal/via stack. For example, if this notebook says logical `m9` is GDS `39/60`, then the process file used by EMX must interpret `39/60` as the intended top metal.

Touchstone numbering follows the YAML port order shown above. With the generated tank defaults, the first four ports are `PP`, `PN`, `SP`, `SN`, so a differential objective usually uses primary ports `[1, 2]` and secondary ports `[3, 4]`. Guard ports, when present, come after the signal ports.


In [ ]:
def emx_command_preview(cfg, gds_path, results_dir, *, proc_file='<absolute-path-to-process-file.proc>', key='<optional-key-or-omit>'):
    nports = max(1, len(cfg.ports))
    touchstone = Path(results_dir) / f'result.s{nports}p'
    internal_args = [f'--internal={port.name},{port.width_um:g}' for port in cfg.ports if port.width_um is not None]
    key_args = [] if key in {None, '', '<optional-key-or-omit>'} else [f'--key={key}']
    freqs_hz = [freq_ghz * 1e9 for freq_ghz in frequency_sweep_ghz(cfg)]
    parts = [
        cfg.emx.executable,
        '--touchstone',
        f'--s-file={touchstone}',
        '--include-command-line',
        '--verbose=2',
        *key_args,
        *internal_args,
        *cfg.emx.extra_args,
        str(gds_path),
        cfg.layout.top_cell,
        proc_file,
        *[f'{freq_hz:.12g}' for freq_hz in freqs_hz],
    ]
    return ' '.join(shlex.quote(str(part)) for part in parts)

print(emx_command_preview(cfg, exported_gds, out_dir / 'emx_preview_results'))


## 6. Make A Real-EMX Local Config

Keep real EMX paths, process keys, license variables, and host-specific setup in ignored local files. The example below writes a local YAML template under `local/`. Edit it on the EMX host before setting `backend: real` for production work.


In [ ]:
real_template = yaml.safe_load(config_path.read_text())
real_template['emx'].update(
    {
        'backend': 'real',
        'executable': 'emx',
        'proc_file': '/absolute/path/to/process_file.proc',
        'env_script': '/absolute/path/to/setup_emx_env.sh',
        'key': None,
        'extra_args': [],
        'timeout_s': 900,
        'retries': 1,
    }
)
real_template_path = out_dir / 'public_dual_core_vco_tank_ring33.real.local.yaml'
real_template_path.write_text(yaml.safe_dump(real_template, sort_keys=False), encoding='utf-8')

print('Wrote local real-EMX template:', real_template_path)
print(yaml.safe_dump({'emx': real_template['emx'], 'ports': real_template['ports']}, sort_keys=False))


A source-safe environment script should set modules, tool paths, and license variables, then return to the caller. Avoid `exit`, `cd`, or commands that deactivate your Python virtual environment.


In [ ]:
setup_script_template = '''#!/usr/bin/env bash
# Source-safe EMX environment setup. Keep this file local and untracked.
# module load <cadence-or-emx-module>
# source <site-cadence-setup.sh>
# export CDS_LIC_FILE=<license-server>
# export LM_LICENSE_FILE=<license-server>
# command -v emx >/dev/null
'''
print(setup_script_template)


## 7. Run `emx-dbs`

The fake backend commands below are safe to run anywhere. For a real EMX host, use the local real-EMX config template from the previous section after replacing the placeholder paths.


In [ ]:
fake_commands = f'''
cd "{repo_root}"
source .venv/bin/activate
python -m pip install -e ".[dev]"

emx-dbs validate-env "{config_path}"
emx-dbs inspect-gds "{config_path}"
emx-dbs preview-input "{config_path}" --output "{out_dir / 'configured_input_preview.png'}"
emx-dbs rasterize "{config_path}"
emx-dbs eval-one "{config_path}"
emx-dbs report "{run_dir}" --summary-only
'''
print(fake_commands)


Run one fake evaluation from Python to confirm the generated GDS, config, EMX runner, Touchstone parser, and objective all agree. This writes artifacts under `local/`.


In [ ]:
fake_run_dir = eval_one(config_path)
summary = generate_report(fake_run_dir, summary_only=True)
print('Fake run dir:', fake_run_dir)
print(json.dumps(summary, indent=2, sort_keys=True, default=str))

run_script = fake_run_dir / 'evaluations' / 'eval_0000' / 'emx' / 'run_emx.sh'
metrics = fake_run_dir / 'evaluations' / 'eval_0000' / 'results' / 'metrics.json'
print('Generated run script:', run_script)
print(run_script.read_text())
print('Metrics:', metrics)
print(metrics.read_text())


## 8. Files Written By This Notebook

The generated public notebook is tracked, but the GDS, YAML, previews, and run outputs below live under ignored `local/` paths.


In [ ]:
for path in sorted(out_dir.rglob('*')):
    if path.is_file():
        print(path.relative_to(repo_root))
